In [ ]:
from datasets import Dataset
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
from copy import deepcopy
import re
from transformers import TrainerCallback
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import StratifiedKFold

In [ ]:
TARGET_COLUMN = 'scheme'
TRAIN_COLUMN = 'text'
MINIBATCH_SIZE = 16
SEED = 42
MAX_TOKENS = 512
DATA_FOLDER = 'C:/Programm/diploma/DataLoad/'
INF_DATA = "inf_8.csv"
CONF_DATA = "conf_8.csv"

In [ ]:
MODEL_NAME = 'DeepPavlov/rubert-base-cased'
# MODEL_NAME = 'ai-forever/ruBert-base'
# MODEL_NAME = 'DeepPavlov/rubert-base-cased-sentence'
# MODEL_NAME = 'ai-forever/ru-en-RoSBERTa'
# MODEL_NAME = "kazzand/ru-longformer-tiny-16384"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
special_tokens_dict = {
    'additional_special_tokens': [
        '<Premise>', '</Premise>',
        '<Conclusion>', '</Conclusion>'
    ]
}

num_added_tokens = tokenizer.add_special_tokens(special_tokens_dict)
print(f"Добавлено {num_added_tokens} новых токенов.")

In [ ]:
inf_df = pd.read_csv(DATA_FOLDER + INF_DATA)
conf_df = pd.read_csv(DATA_FOLDER + CONF_DATA)

In [ ]:
def newClasses(inf_df, conf_df):
    inf = inf_df.copy(deep=True)
    conf = conf_df.copy(deep=True)

    # inference------------------------------------------------------------
    argument_inf = ['DirectAdHominem_Inference',
                    'CircumstantialAdHominem_Inference',
                    'VagueVerbalClassification_Inference',
                    'ArbitraryVerbalClassification_Inference']
    source_inf = ['InconsistentCommitment_Inference',
                'Bias_Inference',]
    thesis_inf = ['CausalSlipperySlope_Inference',
                'VerbalSlipperySlope_Inference',
                'FullSlipperySlope_Inference',
                'PrecedentSlipperySlope_Inference',
                'NegativeConsequences_Inference',
                'FalsificationOfHypothesis_Inference',
                'ExceptionalCase_Inference',
                'FearAppeal_Inference']
    attack_inf = np.concatenate([argument_inf, source_inf, thesis_inf])

    cond = inf['scheme'].isin(attack_inf)
    inf['attack'] = np.where(cond, '1', '0')
    inf.loc[~cond, 'scheme'] = 'Inference'

    # conflict------------------------------------------------------------
    conf['attack'] = '1'
    #------
    conf['scheme'] = 'Conflict'
    return pd.concat([inf, conf], ignore_index=True)

In [ ]:
u_inf_df = newClasses(inf_df, conf_df)

Cделать одинаковую структуру 

In [ ]:
def split2paragraphs(text):
    return [p.strip() for p in re.split(r'\n+', text.strip()) if p.strip()]

def getParagraphIdx(text):
    if not text:
        return []
    # Если переносов нет, считаем весь текст одним абзацем
    if '\n' not in text:
        return [len(text)]
    # Иначе ищем индексы конца абзацев
    matches = list(re.finditer(r'(?:^|\n+)', text))
    indices = [match.end() for match in matches if match.end() < len(text)]
    indices.append(len(text))  # включаем конец текста как последний абзац
    return indices

def updateContext(s="", cont="", col=""):
    if not cont or not s:
        return f"не нашли строку в {col}", cont

    # --- NEW: разбиваем строку на сегменты ---
    segments = [seg.strip() for seg in s.split("<del>") if seg.strip()]
    if not segments:
        return f"не нашли строку в {col}", cont

    # Проверяем, что ВСЕ сегменты найдены в тексте
    if not all(seg in cont for seg in segments):
        return f"не нашли строку в {col}", cont

    # Получаем индексы концов абзацев
    para_idxs = getParagraphIdx(cont)
    if not para_idxs:
        para_idxs = [len(cont)]  # один абзац

    # Определяем границы по минимальному и максимальному сегменту
    starts = [cont.find(seg) for seg in segments]
    ends   = [cont.find(seg) + len(seg) for seg in segments]

    if any(x == -1 for x in starts):
        return f"не нашли строку в {col}", cont

    start_idx = min(starts)
    end_idx   = max(ends)

    # Находим абзац начала и конца
    para_start_index = 0
    for i in range(len(para_idxs)):
        if start_idx < para_idxs[i]:
            para_start_index = i
            break

    para_end_index = len(para_idxs) - 1
    for i in range(len(para_idxs)):
        if end_idx <= para_idxs[i]:
            para_end_index = i
            break

    # Берём текст абзаца целиком
    para_start_pos = 0 if para_start_index == 0 else para_idxs[para_start_index - 1]
    para_end_pos = para_idxs[para_end_index]
    context = cont[para_start_pos:para_end_pos].strip()

    # Отмечаем КАЖДЫЙ сегмент тегами роли
    role = col.split('_')[0]
    tagged_context = context

    # чтобы не портить индексы, заменяем сегменты начиная с самых длинных
    for seg in sorted(segments, key=len, reverse=True):
        tagged_context = tagged_context.replace(
            seg,
            f"<{role}> {seg} </{role}>"
        )

    # удаляем префикс "comment N:" если есть
    tagged_context = re.sub(r'comment\s*\d+\s*:\s*', '', tagged_context)

    return tagged_context, context


#rebuilded
def contex2df(df):
    cp_df = df.copy(deep=True)
    
    for i in cp_df.index:
        exmp = cp_df.iloc[i]
        last_context = None
        last_text = None
        last_col = None
        sim_cols = []

        for column in cp_df.columns:
            role = column.split('_')[0]
            if role not in ['Premise', 'Conclusion']:
                continue

            value = exmp[column]
            if not isinstance(value, str):
                continue

            value = value.strip()

            # Проверяем, входит ли value в предыдущий контекст
            if isinstance(last_context, str) and (value in last_context):
                # Повторно используем контекст
                text, _ = updateContext(s=value, cont=last_text, col=column)
                sim_cols.append(last_col) if sim_cols[-1] != last_col else None
                sim_cols.append(column)
                cp_df.loc[i, sim_cols] = text
            else:
                # Новый контекст
                if isinstance(exmp['comments'], str) and (value in exmp['comments']):
                    text, context = updateContext(s=value, cont=exmp['comments'], col=column)
                elif isinstance(exmp['content'], str) and (value in exmp['content']):
                    text, context = updateContext(s=value, cont=exmp['content'], col=column)
                else:
                    text = f"<{role.lower()}> {value} </{role.lower()}>"
                    context = text

                cp_df.at[i, column] = text
                sim_cols = [column]

            last_context = context
            last_text = text
            last_col = column

    return cp_df


In [ ]:
u_context_inf_df = contex2df(u_inf_df)

In [ ]:
def ensureDot(text):
    text = text.lower()
    text = text.rstrip()  # Убираем пробелы в конце
    if not re.search(r"[a-zA-Zа-яА-Я0-9]$", text):  # Проверяем последний значимый символ
        return text  
    return text + "."  # Добавляем точку

def columnFilter(data, columns_to_keep):
    return data.remove_columns([col for col in data.column_names if col not in columns_to_keep])

def preprocessFunction(example, target_column):
    # Собираем все тексты из колонок
    texts = [
        ensureDot(str(example[col]))
        for col in example.keys()
        if col != target_column and col != 'scheme' and not pd.isna(example[col])
    ]
    
    # Убираем дубликаты, сохраняя порядок
    texts = list(dict.fromkeys(texts))
    
    # Объединяем в одну строку с SEP
    example[TRAIN_COLUMN] = ' [SEP] '.join(texts).strip()
    return example

def tokenizeAndLabeling(datasets, label2id, target_column):
    datasets = [deepcopy(dataset) for dataset in datasets]
    for i in range(len(datasets)):
        datasets[i] = datasets[i].map(
            lambda it: tokenizer(it[TRAIN_COLUMN], truncation=True, padding="max_length", max_length=MAX_TOKENS),
            batched=True, batch_size=MINIBATCH_SIZE
        )
        datasets[i] = datasets[i].add_column(
            'labels',
            [label2id[val] for val in datasets[i][target_column]]
        )
    return datasets

In [ ]:
def prepare_Dataset(df: pd.DataFrame, 
                    target_column='scheme'):
    """
    Подготовка Dataset простой конкатенацией всех столбцов кроме целевого
    """
    # Удаляем ненужные столбцы
    df = df.drop(columns=['uri', 'comments', 'content', 'attacked_arg_uri'], errors='ignore')
    
    # Простая конкатенация всех оставшихся столбцов
    def concatenate_columns(row):
        text_parts = []
        for col in df.columns:
            if col != target_column:  # Исключаем целевой столбец
                val = row[col]
                if pd.notna(val):
                    text_parts.append(str(val))
        return " ".join(text_parts)
    
    # Создаем новый DataFrame с текстом и метками
    processed_df = pd.DataFrame({
        TRAIN_COLUMN: df.apply(concatenate_columns, axis=1),
        target_column: df[target_column]
    })
    
    # Создаем Dataset
    ds = Dataset.from_pandas(processed_df)
    ds = columnFilter(ds, np.unique([target_column, TRAIN_COLUMN]))
    
    return ds

def dfBalance(df, col):
    min_size = df[col].value_counts().min()
    return df.groupby(col).apply(lambda x: x.sample(min_size, random_state=SEED)).reset_index(drop=True)

def load_prepare_data(df, min_n=10, save=False, save_path=None, 
                      target_column=''):
    #убираем схемы которых меньше min_n
    unique_sch = df[target_column].value_counts()
    valid_sch = unique_sch[unique_sch >= min_n].index
    df = df[df[target_column].isin(valid_sch)]
    #делим на выборки
    unique_cnt = df['content'].unique()
    train_val_values, test_values = train_test_split(unique_cnt, test_size=0.15, random_state=SEED)
    train_values, val_values = train_test_split(train_val_values, test_size=0.2, random_state=SEED)

    train_df = df[df['content'].isin(train_values)].reset_index(drop=True) 
    train_df = dfBalance(train_df, target_column)
    val_df = df[df['content'].isin(val_values)].reset_index(drop=True) 
    val_df = dfBalance(val_df, target_column)
    test_df = df[df['content'].isin(test_values)].reset_index(drop=True)   
    train_set, valid_set, test_set = list(map(lambda it: prepare_Dataset(it, target_column=target_column), [train_df, val_df, test_df]))

    if save:
        train_df.to_csv(save_path + '/train.csv', index=False)
        val_df.to_csv(save_path + '/val.csv', index=False)
        test_df.to_csv(save_path + '/test.csv', index=False)

    id_cat = list(range(len(valid_sch)))
    label2id = dict(zip(valid_sch, id_cat))
    id2label = dict(zip(id_cat, valid_sch)) 

    labeled_train_set, labeled_valid_set = tokenizeAndLabeling([train_set, valid_set], label2id, target_column)
    
    return labeled_train_set, labeled_valid_set, test_set, valid_sch, label2id, id2label

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.train_loss = []
        self.eval_loss = []
        self.epochs = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        if "loss" in logs and "eval_loss" not in logs: self.train_loss.append(logs["loss"])
        if "eval_loss" in logs: 
            self.eval_loss.append(logs["eval_loss"])
            self.epochs.append(state.epoch)

    def plot(self):
        import matplotlib.pyplot as plt
        plt.figure(figsize=(8,5))
        plt.plot(self.epochs, self.train_loss, label="Train Loss", lw=2)
        plt.plot(self.epochs, self.eval_loss, label="Val Loss", lw=2)
        plt.xlabel("Epoch"); plt.ylabel("Loss")
        plt.legend(); plt.grid(True); plt.title("Loss over Epochs"); plt.show()

1. Oversample

In [ ]:
def oversample_df(df, target_column):
    counts = df[target_column].value_counts()
    max_count = counts.max()
    dfs = [df[df[target_column] == cls].sample(max_count, replace=True, random_state=SEED) for cls in counts.index]
    return pd.concat(dfs).sample(frac=1, random_state=SEED).reset_index(drop=True)

2. Class weights

In [ ]:
def compute_class_weights_from_dataset(dataset):
    labels = np.array(dataset['labels'])
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    return weights

3. WeighedTrainer

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)

        if class_weights is not None:
            if isinstance(class_weights, np.ndarray):
                class_weights = torch.tensor(class_weights, dtype=torch.float)

            self.class_weights = class_weights.to(self.args.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        # loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss_fct = FocalLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

4. Focal Loss

In [ ]:
class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce_loss = torch.nn.functional.cross_entropy(
            logits, targets, weight=self.weight, reduction='none'
        )
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()


-------------------------------RUN-------------------------------

Подготовка данных и разбиение их по фолдам для стратифицированной классификации

In [ ]:
def cross_validation_load_prepare_data(df, min_n=10, n_splits=5, seed=SEED, max_attempts=50):
    # 1. Фильтр редких схем
    valid_schemes = df['scheme'].value_counts()
    valid_schemes = valid_schemes[valid_schemes >= min_n].index
    df = df[df['scheme'].isin(valid_schemes)].reset_index(drop=True)
    
    print(f"После фильтрации: {len(df)} samples, {df['content'].nunique()} unique contents")
    
    # 2. Глобальные классы
    global_unique_atk = sorted(df['attack'].unique())
    global_unique_sch = sorted(df[df['attack'] == '1']['scheme'].unique())
    
    label2id_atk = {k: i for i, k in enumerate(global_unique_atk)}
    id2label_atk = {i: k for k, i in label2id_atk.items()}
    
    label2id_sch = {k: i for i, k in enumerate(global_unique_sch)}
    id2label_sch = {i: k for k, i in label2id_sch.items()}
    
    # 3. Группировка по content для стратификации
    df_groups = df.groupby('content').agg({
        'attack': lambda x: list(x.unique()),
        'scheme': lambda x: list(x.unique()) if any(x) else []
    }).reset_index()
    
    # Создаем стратификационную метку
    df_groups['has_attack_1'] = df_groups['attack'].apply(lambda x: 1 if '1' in x else 0)
    df_groups['scheme_str'] = df_groups['scheme'].apply(
        lambda x: "|".join(sorted(x)) if x else "none"
    )
    df_groups['stratify'] = df_groups['has_attack_1'].astype(str) + "_" + df_groups['scheme_str']
    
    # Выводим распределение стратификационных групп
    print("\nСтратификационные группы:")
    print(df_groups['stratify'].value_counts())
    
    # 4. Поиск валидных фолдов с использованием стратификации
    for attempt in range(max_attempts):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed + attempt)
        folds = []
        valid_split = True
        
        for fold_idx, (train_ids, val_ids) in enumerate(
                skf.split(df_groups['content'], df_groups['stratify'])):
            
            train_contents = set(df_groups.iloc[train_ids]['content'].values)
            val_contents = set(df_groups.iloc[val_ids]['content'].values)
            
            # Проверка 1: Нет пересечения контентов
            if train_contents.intersection(val_contents):
                print(f"Attempt {attempt}, Fold {fold_idx}: Intersection detected")
                valid_split = False
                break
            
            train_df = df[df['content'].isin(train_contents)].reset_index(drop=True)
            val_df = df[df['content'].isin(val_contents)].reset_index(drop=True)
            
            # Проверка 2: Все attack классы в train
            train_atk = set(train_df['attack'].unique())
            if set(global_unique_atk) - train_atk:
                print(f"Attempt {attempt}, Fold {fold_idx}: Missing attack in train")
                valid_split = False
                break
            
            # Проверка 3: Все attack классы в val
            val_atk = set(val_df['attack'].unique())
            if set(global_unique_atk) - val_atk:
                print(f"Attempt {attempt}, Fold {fold_idx}: Missing attack in val")
                valid_split = False
                break
            
            # Проверка 4: Все scheme классы в train
            train_sch = set(train_df[train_df['attack'] == '1']['scheme'].unique())
            if set(global_unique_sch) - train_sch:
                print(f"Attempt {attempt}, Fold {fold_idx}: Missing scheme in train")
                valid_split = False
                break
            
            # Проверка 5: Все scheme классы в val
            val_sch = set(val_df[val_df['attack'] == '1']['scheme'].unique())
            if set(global_unique_sch) - val_sch:
                print(f"Attempt {attempt}, Fold {fold_idx}: Missing scheme in val")
                valid_split = False
                break
            
            folds.append({
                "fold": fold_idx,
                "train_df": train_df,
                "val_df": val_df,
                "train_contents": train_contents,
                "val_contents": val_contents
            })
        
        if valid_split:
            print(f"\n✅ Found valid split on attempt {attempt}")
            
            # Статистика по фолдам
            for f in folds:
                train_atk = set(f['train_df']['attack'].unique())
                val_atk = set(f['val_df']['attack'].unique())
                train_sch = set(f['train_df'][f['train_df']['attack']=='1']['scheme'].unique())
                val_sch = set(f['val_df'][f['val_df']['attack']=='1']['scheme'].unique())
                
                print(f"\nFold {f['fold']}:")
                print(f"  Train: {len(f['train_df'])} samples, {len(f['train_contents'])} contents")
                print(f"  Val: {len(f['val_df'])} samples, {len(f['val_contents'])} contents")
                print(f"  Attack in train: {sorted(train_atk)}")
                print(f"  Attack in val: {sorted(val_atk)}")
                print(f"  Scheme in train: {sorted(train_sch)}")
                print(f"  Scheme in val: {sorted(val_sch)}")
            
            break
    else:
        raise ValueError(f"Не удалось собрать фолды за {max_attempts} попыток")
    
    # 5. Подготовка Dataset
    prepared_folds = []
    
    for fold_data in folds:
        train_df = fold_data["train_df"]
        val_df = fold_data["val_df"]
        
        # Stage 1: Attack
        train_set_atk = prepare_Dataset(train_df, 'attack')
        val_set_atk = prepare_Dataset(val_df, 'attack')
        
        train_set_atk, val_set_atk = tokenizeAndLabeling(
            [train_set_atk, val_set_atk],
            label2id_atk,
            'attack'
        )
        
        # Stage 2: Scheme
        train_df_sch = train_df[train_df['attack'] == '1'].reset_index(drop=True)
        val_df_sch = val_df[val_df['attack'] == '1'].reset_index(drop=True)
        
        # train_df_sch = oversample_df(train_df_sch, 'scheme')
        
        train_set_sch = prepare_Dataset(train_df_sch, 'scheme')
        val_set_sch = prepare_Dataset(val_df_sch, 'scheme')
        
        train_set_sch, val_set_sch = tokenizeAndLabeling(
            [train_set_sch, val_set_sch],
            label2id_sch,
            'scheme'
        )
        
        prepared_folds.append({
            "fold": fold_data["fold"],
            "train_df": train_df,
            "val_df": val_df,
            "train_atk": train_set_atk,
            "valid_atk": val_set_atk,
            "train_sch": train_set_sch,
            "valid_sch_full": val_set_sch,
            "label2id_atk": label2id_atk,
            "id2label_atk": id2label_atk,
            "label2id_sch": label2id_sch,
            "id2label_sch": id2label_sch,
            "categories_atk": global_unique_atk,
            "categories_sch": global_unique_sch
        })
    
    return prepared_folds

In [ ]:
def run_model_fold_classification_report(
        data_name,
        train_set,
        validation_set,
        tokenizer,
        categories,
        label2id,
        id2label,
        resume_from_checkpoint=False,
        num_train_epochs=3,
        lr=5e-5,
        class_weights=None
    ):

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(categories), id2label=id2label, label2id=label2id
    ).cuda()

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    loss_logger = LossLoggerCallback()

    if class_weights is None:
        class_weights = compute_class_weights_from_dataset(train_set)

    model.resize_token_embeddings(len(tokenizer))

    training_args = TrainingArguments(
        output_dir=f'weights_{data_name}/{MODEL_NAME.split("/")[1]}',
        learning_rate=lr,
        per_device_train_batch_size=MINIBATCH_SIZE,
        per_device_eval_batch_size=MINIBATCH_SIZE,
        num_train_epochs=num_train_epochs,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='best',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        fp16=True,
        seed=SEED,
        save_total_limit=1
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_set,
        eval_dataset=validation_set,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        callbacks=[loss_logger]
    )

    trainer.train(resume_from_checkpoint=resume_from_checkpoint)

    preds_output = trainer.predict(validation_set)
    preds_idx = np.argmax(preds_output.predictions, axis=1)
    labels_idx = preds_output.label_ids

    # Названия классов для глобального накопления
    preds_names = [id2label[i] for i in preds_idx]
    labels_names = [id2label[i] for i in labels_idx]

    labels_present_idx = np.unique(np.concatenate([labels_idx, preds_idx]))

    report = classification_report(
        labels_idx,
        preds_idx,
        labels=labels_present_idx,
        target_names=[id2label[l] for l in labels_present_idx],
        digits=4,
        zero_division=0
    )

    report_dict = classification_report(
        labels_idx,
        preds_idx,
        labels=labels_present_idx,
        target_names=[id2label[l] for l in labels_present_idx],
        output_dict=True,
        zero_division=0
    )

    return (
        report_dict['macro avg']['precision'],
        report_dict['macro avg']['recall'],
        report_dict['macro avg']['f1-score'],
        report,
        preds_idx,
        labels_idx,
        preds_names,
        labels_names
    )
    
def run_crossval_2stage(
        data_name, df, tokenizer,
        n_splits=5, percentile=75,
        resume_from_checkpoint=False,
        num_train_epochs=5, lr=1e-4, min_n=10):

    # Загружаем и подготавливаем данные с фолдами
    folds = cross_validation_load_prepare_data(df, n_splits=n_splits, min_n=min_n)

    atk_f1 = []
    sch_f1 = []

    all_preds_atk = []
    all_labels_atk = []

    all_preds_sch = []
    all_labels_sch = []

    for f in folds:
        print(f"\n===== FOLD {f['fold']} =====")

        # ======================
        # STAGE 1: ATTACK
        # ======================
        print("\n--- ATTACK ---")

        # вычисляем class weights для Attack
        counts_atk = np.array([np.sum(f["train_df"]["attack"]==c) for c in f["categories_atk"]])
        class_weights_atk = np.sum(counts_atk) / (len(counts_atk) * counts_atk)
        
        p, r, f1, rep, preds, labels, preds_names, labels_names = \
            run_model_fold_classification_report(
                f"{data_name}_atk_{f['fold']}",
                f["train_atk"],
                f["valid_atk"],
                tokenizer,
                f["categories_atk"],
                f["label2id_atk"],
                f["id2label_atk"],
                resume_from_checkpoint=resume_from_checkpoint,
                num_train_epochs=5,
                lr=lr,
                class_weights=class_weights_atk
            )

        print(rep)
        atk_f1.append(f1)

        all_preds_atk.extend(preds_names)
        all_labels_atk.extend(labels_names)

        val_df = f["val_df"]
        original_attack_labels = val_df['attack'].values

        # ======================
        # STAGE 2: SCHEME
        # ======================
        print("\n--- SCHEME ---")

        val_df = f["val_df"]

        attack_indices = np.where(preds == 1)[0]
        # attack_indices = np.where(original_attack_labels == '1')[0]

        if len(attack_indices) == 0:
            print("No predicted attacks")
            continue

        val_subset = val_df.iloc[attack_indices]
        val_subset = val_subset[val_subset['attack'] == '1']

        if len(val_subset) == 0:
            print("No correct attack predictions")
            continue

        # Oversample Stage 2 (Scheme)
        train_df_sch = oversample_df(f["train_df"][f["train_df"]["attack"]=='1'], target_column='scheme')
        # train_df_sch = f["train_df"][f["train_df"]["attack"]=='1']

        train_set_sch = prepare_Dataset(train_df_sch, target_column='scheme')
        val_dataset = prepare_Dataset(val_subset, target_column='scheme')

        # Убираем несоответствия классов между train и val
        unique_sch = pd.concat([train_df_sch['scheme'], val_subset['scheme']]).unique()
        label2id_sch = {k:i for i,k in enumerate(unique_sch)}
        id2label_sch = {i:k for k,i in label2id_sch.items()}

        train_set_sch, val_dataset = tokenizeAndLabeling([train_set_sch, val_dataset], label2id_sch, 'scheme')

        # class weights для Scheme
        counts_sch = np.array([np.sum(train_df_sch['scheme']==c) for c in unique_sch])
        class_weights_sch = np.sum(counts_sch) / (len(counts_sch) * counts_sch)

        p, r, f1, rep, preds_sch, labels_sch, preds_names, labels_names = \
            run_model_fold_classification_report(
                f"{data_name}_sch_{f['fold']}",
                train_set_sch,
                val_dataset,
                tokenizer,
                unique_sch,
                label2id_sch,
                id2label_sch,
                resume_from_checkpoint=resume_from_checkpoint,
                num_train_epochs=num_train_epochs,
                lr=lr,
                class_weights=class_weights_sch
            )

        print(rep)
        sch_f1.append(f1)

        all_preds_sch.extend(preds_names)
        all_labels_sch.extend(labels_names)

    # ======================
    # FINAL REPORT
    # ======================
    print("\n===== FINAL =====")

    print("\nATTACK F1 по фолдам:", atk_f1)
    print("mean F1 по фолдам:", np.mean(atk_f1))
    print(f"{percentile}th percentile:", np.percentile(atk_f1, percentile))

    print("\nSCHEME F1 по фолдам:", sch_f1)
    if len(sch_f1):
        print("mean F1 по фолдам:", np.mean(sch_f1))
        print(f"{percentile}th percentile:", np.percentile(sch_f1, percentile))
    else:
        print("No scheme F1 scores available")

    # ======================
    # GLOBAL REPORTS
    # ======================
    print("\n===== GLOBAL ATTACK REPORT =====")
    global_atk_report = classification_report(
        all_labels_atk,
        all_preds_atk,
        digits=4,
        zero_division=0
    )
    print(global_atk_report)

    print("\n===== GLOBAL SCHEME REPORT =====")
    if len(all_labels_sch):
        global_sch_report = classification_report(
            all_labels_sch,
            all_preds_sch,
            digits=4,
            zero_division=0
        )
        print(global_sch_report)
    else:
        global_sch_report = "No scheme predictions"
        print(global_sch_report)

    return {
        "attack_f1_per_fold": atk_f1,
        "scheme_f1_per_fold": sch_f1,
        "global_attack_report": global_atk_report,
        "global_scheme_report": global_sch_report
    }


-------------------------------RUN-------------------------------

In [ ]:
results = run_crossval_2stage(
    data_name=INF_DATA,
    df=u_context_inf_df,
    tokenizer=tokenizer,
    n_splits=5,
    percentile=75, 
    resume_from_checkpoint=False, 
    num_train_epochs=20,
    lr=5e-5,
    min_n=20
)